# 🧩 Mini-Lab: Settings from Environment

**Module: Trust but Verify: Validation, Config, Errors, Logs** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** the three-layer configuration pattern: hard-coded defaults → `.env` file → environment variables
2. **Build** a settings object that loads values from defaults, a `.env` file, and environment overrides
3. **Explain** why `.env` files must be git-ignored and secrets must never be hard-coded in source

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Layered Configuration | Defaults live in code, environments override them at deploy time, and secrets never go in Git, so you can change behavior safely without editing source. |
| Secrets Hygiene | Secrets hygiene is the habit of keeping passwords, API keys, and tokens out of code and logs, sharing them safely, and rotating them regularly so one mistake doesn’t become a breach. |
| Version-Control Basics | Init a repo, ignore generated files, commit small changes, and tag v0 to v5 as known good checkpoints you can reset to safely. |


## Why Layered Configuration?

Hard-coding settings in source code is fragile and dangerous:

| Problem | Example |
|---------|---------|
| **Leaked secrets** | `api_key = "sk-abc123"` pushed to GitHub |
| **Inflexible deployments** | Changing a timeout means editing code and redeploying |
| **No per-environment control** | Dev and prod use the same values |

The **layered configuration** pattern solves all three. Settings come from three sources, each overriding the last:

```
Layer 1 (lowest priority):  Hard-coded DEFAULTS in code
Layer 2:                    .env file  (local dev, git-ignored)
Layer 3 (highest priority): Real ENVIRONMENT VARIABLES  (CI/CD, containers)
```

This means you always have a working default, developers can customize via `.env`, and production overrides via real environment variables win.

## Step 1 — Install `python-dotenv`

`python-dotenv` reads key-value pairs from a `.env` file into environment variables. It's the standard lightweight tool for this pattern.

In [1]:
# !pip install -q python-dotenv

## Step 2 — Create a `.env` File

In a real project you'd create `.env` by hand in your project root. Here we'll write one programmatically so the lab is self-contained.

> **Key rule:** This file is for *local development only*. It must **never** be committed to version control.

In [2]:
from pathlib import Path

env_content = """# Local development settings - DO NOT COMMIT
OPENAI_API_KEY=sk-fake-dev-key-12345
MODEL_NAME=gpt-4o-mini
REQUEST_TIMEOUT=45
"""

Path(".env").write_text(env_content.strip(), encoding="utf-8")
print("Created .env file:")
print(Path(".env").read_text(encoding="utf-8"))

Created .env file:
# Local development settings - DO NOT COMMIT
OPENAI_API_KEY=sk-fake-dev-key-12345
MODEL_NAME=gpt-4o-mini
REQUEST_TIMEOUT=45


## Step 3 — Add `.env` to `.gitignore`

Before doing anything else, ensure `.env` is in your `.gitignore`. If the file doesn't exist, we create one. If it does, we append the rule only if it's missing.

> **Why before anything else?** The moment `.env` exists on disk, a careless `git add .` could commit it. Adding the ignore rule *first* prevents that.

In [3]:
gitignore_path = Path(".gitignore")

# Read existing content (or start empty)
existing = gitignore_path.read_text(encoding="utf-8") if gitignore_path.exists() else ""

# Only append if not already present
if ".env" not in existing.splitlines():
    with open(gitignore_path, "a", encoding="utf-8") as f:
        f.write("\n.env\n")
    print("✅ Added .env to .gitignore")
else:
    print("✅ .env already in .gitignore")

✅ .env already in .gitignore


## Step 4 — Build the Layered Settings Object

We define a `Settings` dataclass with sensible defaults (Layer 1), then load values from the `.env` file (Layer 2), and finally allow real environment variables to override everything (Layer 3).

The function `load_settings()` implements all three layers in order.

In [4]:
import os
from dataclasses import dataclass
from dotenv import load_dotenv


@dataclass
class Settings:
    """Application settings built from layered configuration."""
    openai_api_key: str
    model_name: str
    request_timeout: int
    max_retries: int


# --- Layer 1: Hard-coded defaults ---
DEFAULTS = {
    "OPENAI_API_KEY": "",          # no default — must be provided
    "MODEL_NAME": "gpt-4o-mini",
    "REQUEST_TIMEOUT": "30",        # env vars are always strings
    "MAX_RETRIES": "3",
}


def load_settings(env_path: str = ".env") -> Settings:
    """Load settings: defaults → .env file → environment variables."""

    # --- Layer 2: .env file (does NOT override already-set env vars) ---
    load_dotenv(env_path, override=False, encoding="utf-8")

    # --- Layer 3: os.environ (real env vars) always wins ---
    # We read from os.environ, falling back to our defaults.
    def get(key: str) -> str:
        return os.environ.get(key, DEFAULTS[key])

    api_key = get("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("OPENAI_API_KEY is required but not set in .env or environment")

    return Settings(
        openai_api_key=api_key,
        model_name=get("MODEL_NAME"),
        request_timeout=int(get("REQUEST_TIMEOUT")),
        max_retries=int(get("MAX_RETRIES")),
    )

## Step 5 — See the Layers in Action

Let's load settings and inspect which values came from which layer.

In [5]:
settings = load_settings()
print(settings)

Settings(openai_api_key='sk-fake-dev-key-12345', model_name='gpt-4o-mini', request_timeout=45, max_retries=3)


Let's trace where each value came from:

| Setting | Value | Source |
|---------|-------|--------|
| `openai_api_key` | `sk-fake-dev-key-12345` | **Layer 2** — `.env` file |
| `model_name` | `gpt-4o-mini` | **Layer 2** — `.env` (same as default, but set in `.env`) |
| `request_timeout` | `45` | **Layer 2** — `.env` overrode the default of `30` |
| `max_retries` | `3` | **Layer 1** — default (not in `.env`, not in environment) |

## Step 6 — Layer 3 Overrides Everything

Real environment variables (Layer 3) take the highest priority. Let's simulate setting one and see it override the `.env` value.

In [6]:
# Simulate a real env var (as CI/CD or Docker would set it)
os.environ["REQUEST_TIMEOUT"] = "120"

settings = load_settings()
print(f"request_timeout = {settings.request_timeout}")  # 120, not 45
print("↑ Real env var (Layer 3) overrode .env (Layer 2)")

# Clean up so it doesn't affect later cells
del os.environ["REQUEST_TIMEOUT"]

request_timeout = 120
↑ Real env var (Layer 3) overrode .env (Layer 2)


## Step 7 — Secrets Hygiene: What NOT to Do

Let's see the **wrong** way and the **right** way side by side.

In [7]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

md("""
### ❌ Never do this

```python
# Hard-coded in source code — gets committed to git!
api_key = "sk-real-secret-key-abc123"
```

```python
# Printing or logging secrets
print(f"Using key: {settings.openai_api_key}")
logger.info(f"API key = {api_key}")
```

### ✅ Do this instead

```python
# Load from environment
settings = load_settings()

# Log only a safe suffix
print(f"API key loaded: ...{settings.openai_api_key[-4:]}")
```

### ✅ Always git-ignore your `.env`

```
# .gitignore
.env
```
""")


### ❌ Never do this

```python
# Hard-coded in source code — gets committed to git!
api_key = "sk-real-secret-key-abc123"
```

```python
# Printing or logging secrets
print(f"Using key: {settings.openai_api_key}")
logger.info(f"API key = {api_key}")
```

### ✅ Do this instead

```python
# Load from environment
settings = load_settings()

# Log only a safe suffix
print(f"API key loaded: ...{settings.openai_api_key[-4:]}")
```

### ✅ Always git-ignore your `.env`

```
# .gitignore
.env
```


## Step 8 — Fail Loudly on Missing Secrets

Our `load_settings()` already raises a `ValueError` if the API key is empty. Let's verify that it works — a missing secret should fail at startup, not silently halfway through a pipeline.

In [8]:
# Write a .env without the API key
Path(".env").write_text("MODEL_NAME=gpt-4o-mini\n", encoding="utf-8")

# Remove from environment too, so no layer provides it
os.environ.pop("OPENAI_API_KEY", None)

try:
    bad_settings = load_settings()
except ValueError as exc:
    print(f"✅ Caught expected error: {exc}")

# Restore the original .env for any re-runs
Path(".env").write_text(env_content.strip(), encoding="utf-8")

✅ Caught expected error: OPENAI_API_KEY is required but not set in .env or environment


123

## Cleanup

In [9]:
# Remove the demo files so they don't linger
Path(".env").unlink(missing_ok=True)
print("Removed .env demo file")

Removed .env demo file


## 🎯 Summary

### Key Takeaways

1. **Layered configuration** — build settings from three layers (defaults → `.env` → environment variables) so every value has a sensible fallback and production can override without code changes
2. **Secrets hygiene** — never hard-code API keys in source; load them from environment variables and log only safe suffixes
3. **Git-ignore `.env`** — add `.env` to `.gitignore` *before* you can accidentally commit it; this is the single most important version-control habit for secrets
4. **Fail loudly on missing secrets** — validate required settings at startup so a missing key causes a clear error, not a silent failure mid-pipeline